In [ ]:
import threading
import time

def func():
    print('ran')
    time.sleep(1)
    print("done")
    time.sleep(.85)
    print("now done")

x = threading.Thread(target = func)
x.start()
time.sleep(.9)
print("finally")


In [ ]:
import bisect
class DB:

    # key, field, value

    def __init__(self):
        self.ts = 0
        self.data = {}
        self.lock = threading.RLock()

    def _next(self):
        self.ts += 1
        return self.ts

    def set(self, value, key, field):
        self._set(value, key, field)
        return ""

    def _set(self, value, key, field, ts=None, expiry=None):
        if key not in self.data:
            self.data[key] = {}
        if field not in self.data[key]:
            self.data[key][field] = []
        if ts == None:
            ts = self._next()
        self.data[key][field].append((ts, value, expiry))

    def get(self, key, field, ts):
        res = self._latest(ts, key , field)
        return res[0] if res else ""

    def _latest(self, key, field, query_ts):
        if key not in self.data or field not in self.data[key]:
            return None
        versions = self.data[key][field]
        i = bisect.bisect_right([v[0] for v in versions], query_ts) - 1
        if i < 0:
            return None
        ts, val, exp = versions[i]
        if val is None:
            return None
        if exp is not None and exp <= query_ts:
            return None
        return (val, exp)
    
    def _backup(self, ts):
        snap = self._snapshot(ts)
        self.backups.append((ts,snap))

    def _snapshot(self, query_ts):
        snap = {}
        for key, fields, in self.data.items():
            for field, versions in fields.items():
                keep = [[(ts, val, exp) for (ts, val, exp) in versions if ts <= query_ts]]
                if keep:
                    snap.setdefault(key,{})[field] = keep
        return snap

